# Feature Enineering
From previous notebook, Our best baseline model with trying different models and preprocessing techniques is LightBGM with class weight and one hot It gives us 0.7802 ROC AUC score. Now we will try to improve our model performance by doing feature engineering.

In this notebbok, apply your domain knowledge to create new features that can help the model learn better. This is where you can get creative and experiment with different feature engineering techniques.

### Experimentation Setup 
Before feature eninerring so our work of repitative work will reduce. and model evaluate on same metrics, same preprocessing pipeline, and same train test split. So we can easily compare the performance of different feature engineering techniques and see which one works best for our dataset. 

In [10]:
import pandas as pd

train = pd.read_csv("../data/processed/train.csv")
test  = pd.read_csv("../data/processed/test.csv")

X_train = train.drop(columns=["Churn"])
y_train = train["Churn"]

X_test  = test.drop(columns=["Churn"])
y_test  = test["Churn"]

# Drop CustomerID column as it is not useful for modeling
X_train = X_train.drop(columns=["CustomerID"])
X_test  = X_test.drop(columns=["CustomerID"])

In [12]:
def get_feature_lists(df):
    numeric = df.select_dtypes(include=["int64","float64"]).columns.tolist()
    categorical = df.select_dtypes(include=["object"]).columns.tolist()
    return numeric, categorical

In [22]:
# for basekine preprocessing pipeline
numeric_features = X_train.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

In [23]:
# Preprocessing pipeline
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [24]:
# LightBGM model
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline

lightBGM = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(
        class_weight="balanced",
        random_state=42
    ))
])

In [25]:
# Evalution Function
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

def evaluate_model(model, X_train, y_train, X_test, y_test):
    
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict_proba(X_train)[:,1]
    y_test_pred  = model.predict_proba(X_test)[:,1]
    
    train_auc = roc_auc_score(y_train, y_train_pred)
    test_auc  = roc_auc_score(y_test, y_test_pred)
    
    print(f"Train ROC-AUC: {train_auc:.4f}")
    print(f"Test ROC-AUC : {test_auc:.4f}")
    
    y_test_class = (y_test_pred >= 0.5).astype(int)
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_test_class))
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_test_class))
    
    return test_auc

After any feature improvemnt:-

1. create feature with function name and set to X_train_fe and X_test_fe, then evaluate model with same code as before to see the performance improvement. For ex.
- X_train_fe = create_features(X_train)
- X_test_fe  = create_features(X_test)

2. Define  new features
- numeric_features, categorical_features = get_feature_lists(X_train_fe)

3. Then call evaluate_model function to see the performance improvement.  It will handle all the preprocessing and model training and evaluation.
- evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

## Feaure Enginerring

In [26]:
def base_features(df):
    return df.copy()

In [27]:
X_train_fe = base_features(X_train)
X_test_fe  = base_features(X_test)
numeric_features, categorical_features = get_feature_lists(X_train_fe)
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011023 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.8490
Test ROC-AUC : 0.7802

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.75      0.82      8255
           1       0.36      0.66      0.46      1745

    accuracy                           0.74     10000
   macro avg       0.64      0.70      0.64     10000
weighted avg       0.82      0.74      0.76     10000


Confusion Matrix:
[[6209 2046]
 [ 600 1145]]


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


np.float64(0.7802335651398216)

1. From EDA we decide to do feature enginerring for Engagement score, Tenure buckets, Product count,create feature from them


In [28]:
# 1. From EDA we decide to do feature enginerring for Engagement score, Tenure buckets, Product count,create feature from them
def create_features(df, use_engagement=False, use_product=False, use_financial=False):
    
    df = df.copy()
    
    if use_engagement:
        df["Total_Digital_Engagement"] = (
            df["Mobile_App_Logins"].fillna(0)
            + df["UPI_Transactions"].fillna(0)
            + df["NetBanking_Usage"].fillna(0)
        )
    
    if use_product:
        df["Total_Products"] = (
            df["Has_Credit_Card"]
            + df["Has_Loan"]
            + df["Has_Insurance"]
        )
    
    if use_financial:
        df["Balance_Shock_Flag"] = (
            df["Balance_Change_Ratio"] < -0.20
        ).astype(int)
    
    return df

In [29]:
X_train_fe = create_features(X_train, use_engagement=True, use_product=True, use_financial=False)
X_test_fe  = create_features(X_test, use_engagement=True, use_product=True, use_financial=False)

In [30]:
numeric_features, categorical_features = get_feature_lists(X_train_fe)
print("Numeric Features:", numeric_features)
print("Categorical Features:", categorical_features)

Numeric Features: ['Age', 'Annual_Income', 'Credit_Score', 'Tenure_Months', 'Avg_Monthly_Balance', 'Balance_Change_Ratio', 'Mobile_App_Logins', 'UPI_Transactions', 'ATM_Withdrawals', 'NetBanking_Usage', 'Complaint_Tickets', 'Call_Center_Interactions', 'Has_Credit_Card', 'Has_Loan', 'Has_Insurance', 'Total_Digital_Engagement', 'Total_Products']
Categorical Features: ['Gender', 'City', 'Employment_Type']


In [31]:
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006476 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.8490
Test ROC-AUC : 0.7802

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.75      0.82      8255
           1       0.36      0.66      0.46      1745

    accuracy                           0.74     10000
   macro avg       0.64      0.70      0.64     10000
weighted avg       0.82      0.74      0.76     10000


Confusion Matrix:
[[6209 2046]
 [ 600 1145]]


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


np.float64(0.7802335651398216)

| Feature    | Train | Test         | Conclusion       |
| ------- | ----------------- |------------------ |------------------ |
| Enagmenst + product Score + fininace | 0.8490    | 0.7802   | This is mid overfitting, they add flexibility not generalization, LightBGM already captures this time of summation feature. |
| Ratio features | 0.8490    | 0.7802   | Same as previous no chnage in score|
| Tenure buckets | 0.8490    | 0.7802   | No change in score |
| City Tier | 0.8490    | 0.7802   | No change in score |

2. now try ratio features

In [32]:
def add_ratio_features(df):
    df = df.copy()
    
    # 1️⃣ Engagement intensity
    df["Total_Digital_Engagement"] = (
        df["Mobile_App_Logins"].fillna(0)
        + df["UPI_Transactions"].fillna(0)
        + df["NetBanking_Usage"].fillna(0)
    )
    
    df["Engagement_per_Tenure"] = (
        df["Total_Digital_Engagement"] / (df["Tenure_Months"] + 1)
    )
    
    # 2️⃣ Complaint intensity
    df["Complaint_per_Tenure"] = (
        df["Complaint_Tickets"].fillna(0) / (df["Tenure_Months"] + 1)
    )
    
    # 3️⃣ Financial ratios
    df["Balance_to_Income_Ratio"] = (
        df["Avg_Monthly_Balance"] / (df["Annual_Income"] + 1)
    )
    
    df["Balance_per_Tenure"] = (
        df["Avg_Monthly_Balance"] / (df["Tenure_Months"] + 1)
    )
    
    # 4️⃣ Product penetration
    df["Total_Products"] = (
        df["Has_Credit_Card"]
        + df["Has_Loan"]
        + df["Has_Insurance"]
    )
    
    df["Products_per_Tenure"] = (
        df["Total_Products"] / (df["Tenure_Months"] + 1)
    )
    
    return df

In [33]:
X_train_fe = add_ratio_features(X_train)
X_test_fe  = add_ratio_features(X_test)
numeric_features, categorical_features = get_feature_lists(X_train_fe)
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006576 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.8490
Test ROC-AUC : 0.7802

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.75      0.82      8255
           1       0.36      0.66      0.46      1745

    accuracy                           0.74     10000
   macro avg       0.64      0.70      0.64     10000
weighted avg       0.82      0.74      0.76     10000


Confusion Matrix:
[[6209 2046]
 [ 600 1145]]


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


np.float64(0.7802335651398216)

### From confusion matrix
Interpretation:

2046 = False Positives 
→ Bank spends retention budget unnecessarily

600 = False Negatives
→ Bank loses actual churners

In [34]:
import numpy as np

y_probs = lightBGM.predict_proba(X_test_fe)[:,1]

for t in [0.4, 0.5, 0.6, 0.7]:
    y_pred = (y_probs >= t).astype(int)
    print(f"\nThreshold: {t}")
    print(confusion_matrix(y_test, y_pred))


Threshold: 0.4
[[4968 3287]
 [ 386 1359]]

Threshold: 0.5
[[6209 2046]
 [ 600 1145]]

Threshold: 0.6
[[7098 1157]
 [ 812  933]]

Threshold: 0.7
[[7663  592]
 [1055  690]]


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



## 📊 Threshold Comparison

#### 🔎 Threshold = 0.4 (Aggressive)
[[4968 3287]
[ 386 1359]]
- TP ↑ (1359) → More churners detected  
- FN ↓ (386) → Fewer churners missed  
- FP ↑↑ (3287) → High retention cost  
👉 Good when **losing churners is very expensive**  
👉 Bank spends heavily on retention campaigns  

#### 🔎 Threshold = 0.5 (Balanced – Default)
[[6209 2046]
[ 600 1145]]
- Balanced trade-off  
- FP = 2046  
- FN = 600  
👉 Moderate retention cost  
👉 Moderate churn loss  

#### 🔎 Threshold = 0.6 (Conservative)
[[7098 1157]
[ 812 933]]
- FP drops sharply (2046 → 1157)  
- FN increases (600 → 812)  
👉 Bank saves retention cost  
👉 More churners are missed  

## ##🔎 Threshold = 0.7 (Very Conservative)
[[7663 592]
[1055 690]]
- Very low FP (592)  
- High FN (1055)  

👉 Cheap retention strategy  
👉 High customer loss risk  

# 🏦 Business Decision Example

Assume:

- Retention campaign cost = ₹500 per customer  
- Losing a churned customer = ₹10,000 lifetime value  


#### 📌 Threshold = 0.5

- FP cost = 2046 × 500 = ₹10,23,000  
- FN loss = 600 × 10,000 = ₹60,00,000  
- **Total impact = ₹70,23,000**

####  📌 Threshold = 0.6

- FP cost = 1157 × 500 = ₹5,78,500  
- FN loss = 812 × 10,000 = ₹81,20,000  
- **Total impact = ₹86,98,500**

### ✅ Conclusion

In this scenario, **Threshold = 0.5** is better than 0.6  
because total financial impact is lower.


There is no mathematically “best” threshold.

There is only:

> **Best threshold for business objective.**

Threshold should always be selected based on:
- Retention cost
- Customer lifetime value
- Risk tolerance


### Threshold Analysis Interpretation

Lowering threshold increases churn recall but significantly increases false positives, leading to higher retention campaign cost.

Increasing threshold reduces false positives but increases missed churners, potentially leading to higher customer lifetime value loss.

For balanced tradeoff between retention cost and churn loss, threshold = 0.5 was selected for deployment. However, threshold can be adjusted dynamically based on business strategy.

# 3. New feature : Tenure buckets

In [35]:
X_train["Tenure_Months"].value_counts()

Tenure_Months
134    463
146    446
132    445
137    444
136    443
      ... 
231      3
239      1
232      1
238      1
235      1
Name: count, Length: 238, dtype: int64

In [38]:
def add_tenure_bucket(df):
    df = df.copy()
    
    df["Tenure_Bucket"] = pd.cut(
        df["Tenure_Months"],
        bins=[-1, 24, 60, 120, 240],
        labels=["0-2 yrs", "2-5 yrs", "5-10 yrs", "10+ yrs"]
    )
    df["Is_New"] = (df["Tenure_Months"] <= 6).astype(int)
    df["Is_Loyal"] = (df["Tenure_Months"] >= 60).astype(int)
    
    return df

In [40]:
X_train_fe = add_tenure_bucket(X_train)
X_test_fe  = add_tenure_bucket(X_test)
numeric_features, categorical_features = get_feature_lists(X_train_fe)
print(numeric_features)
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)


['Age', 'Annual_Income', 'Credit_Score', 'Tenure_Months', 'Avg_Monthly_Balance', 'Balance_Change_Ratio', 'Mobile_App_Logins', 'UPI_Transactions', 'ATM_Withdrawals', 'NetBanking_Usage', 'Complaint_Tickets', 'Call_Center_Interactions', 'Has_Credit_Card', 'Has_Loan', 'Has_Insurance', 'Is_New', 'Is_Loyal']
[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009927 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.8490
Test ROC-AUC : 0.7802

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.75      0.82      8255
           1       0.36      0.66      0.46      1745

    accuracy                           0.74     10000
   macro avg       0.64      0.70      0.64     10000
weighted avg       0.82      0.74      0.76     10000


Confusion Matrix:
[[6209 2046]
 [ 600 1145]]


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


np.float64(0.7802335651398216)

# New Feature - City Tier

In [41]:
X_train["City"].value_counts()

City
Mumbai              3063
Delhi               2992
Bengaluru           2700
Chennai             2364
Hyderabad           2294
Pune                1955
Kolkata             1944
Ahmedabad           1687
Surat               1382
Jaipur              1310
Lucknow             1049
Indore               999
Thane                998
Kanpur               992
Nagpur               968
Agra                 691
Faridabad            682
Patna                678
Pimpri-Chinchwad     676
Ghaziabad            671
Visakhapatnam        670
Ludhiana             665
Nashik               664
Vadodara             644
Bhopal               643
Varanasi             576
Rajkot               520
Guwahati             515
Amritsar             511
Meerut               503
Ranchi               502
Coimbatore           491
Madurai              483
Srinagar             482
Howrah               474
Navi Mumbai          333
Kochi                325
Mysuru               314
Chandigarh           300
Noida               

In [42]:
X_train["City"].unique()

array(['Chennai', 'Amritsar', 'Coimbatore', 'Agra', 'Madurai',
       'Ahmedabad', 'Kanpur', 'Bengaluru', 'Guwahati', 'Kochi',
       'Faridabad', 'Lucknow', 'Meerut', 'Vadodara', 'Delhi', 'Hyderabad',
       'Nagpur', 'Pimpri-Chinchwad', 'Howrah', 'Bhopal', 'Kolkata',
       'Mumbai', 'Surat', 'Thane', 'Srinagar', 'Nashik', 'Varanasi',
       'Ludhiana', 'Pune', 'Noida', 'Jaipur', 'Indore', 'Ranchi',
       'Visakhapatnam', 'Ghaziabad', 'Rajkot', 'Navi Mumbai', 'Mysuru',
       'Chandigarh', 'Patna'], dtype=object)

In [43]:
tier_1 = [
    "Mumbai", "Delhi", "Bengaluru", "Hyderabad",
    "Chennai", "Kolkata", "Pune", "Ahmedabad"
]

tier_2 = [
    "Jaipur", "Surat", "Lucknow", "Kanpur", "Nagpur",
    "Indore", "Thane", "Bhopal", "Visakhapatnam",
    "Pimpri-Chinchwad", "Patna", "Vadodara",
    "Ghaziabad", "Ludhiana", "Agra",
    "Nashik", "Faridabad", "Coimbatore",
    "Madurai", "Mysuru", "Rajkot",
    "Chandigarh", "Noida"
]

# Remaining cities will automatically become Tier_3

In [44]:
def add_city_tier(df):
    df = df.copy()
    
    def map_tier(city):
        if city in tier_1:
            return "Tier_1"
        elif city in tier_2:
            return "Tier_2"
        else:
            return "Tier_3"
    
    df["City_Tier"] = df["City"].apply(map_tier)
    
    return df

In [45]:
X_train_fe = add_city_tier(X_train_fe)
X_test_fe  = add_city_tier(X_test_fe)
numeric_features, categorical_features = get_feature_lists(X_train_fe)
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016022 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.8490
Test ROC-AUC : 0.7802

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.75      0.82      8255
           1       0.36      0.66      0.46      1745

    accuracy                           0.74     10000
   macro avg       0.64      0.70      0.64     10000
weighted avg       0.82      0.74      0.76     10000


Confusion Matrix:
[[6209 2046]
 [ 600 1145]]


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


np.float64(0.7802335651398216)

In [46]:
# Check with city tier coulumn and drop city column
X_train_fe = add_city_tier(X_train)
X_test_fe  = add_city_tier(X_test)

# Optional test:
X_train_fe = X_train_fe.drop(columns=["City"])
X_test_fe  = X_test_fe.drop(columns=["City"])

numeric_features, categorical_features = get_feature_lists(X_train_fe)
evaluate_model(lightBGM, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 6981, number of negative: 33019
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007666 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1550
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train ROC-AUC: 0.8490
Test ROC-AUC : 0.7802

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.75      0.82      8255
           1       0.36      0.66      0.46      1745

    accuracy                           0.74     10000
   macro avg       0.64      0.70      0.64     10000
weighted avg       0.82      0.74      0.76     10000


Confusion Matrix:
[[6209 2046]
 [ 600 1145]]


C:\Users\vedik\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


np.float64(0.7802335651398216)

| Feature    | Train | Test         | Conclusion       |
| ------- | ----------------- |------------------ |------------------ |
| Enagmenst + product Score + fininace | 0.8490    | 0.7802   | This is mid overfitting, they add flexibility not generalization, LightBGM already captures this time of summation feature. |
| Ratio features | 0.8490    | 0.7802   | Same as previous no chnage in score|
| Tenure buckets | 0.8490    | 0.7802   | No change in score |
| City Tier | 0.8490    | 0.7802   | No change in score |

Already selected LightBGM model captureees all pattern well so feature engineering not perform very well. So we can see that our feature engineering techniques did not improve the model performance. This is a common scenario in machine learning, where not all feature engineering techniques will lead to performance improvement. It is important to experiment and evaluate different techniques to find the ones that work best for your dataset and model.

Next we move to hyperparameter tuning to see if we can improve our model performance.